# 09 — Transformers con HuggingFace

**Level 0 — Fundamentos Software & IA**

Clasificación de sentimiento con un modelo real (DistilBERT fine-tuned).
Empezamos con `pipeline` (todo listo) y después abrimos la caja:
tokenizer → modelo → logits → probabilidades.

In [1]:
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F

clasificador = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
)
print(f"   Pipeline listo: {clasificador.model.__class__.__name__}")

/workspaces/student-ai/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 1371.36it/s]

   Pipeline listo: DistilBertForSequenceClassification


## Clasificar textos con pipeline

In [2]:
textos = [
    "This movie was absolutely fantastic! I loved every minute.",
    "This is the worst film I have ever seen in my entire life.",
    "The acting was okay but the plot was confusing.",
    "I am not sure how to feel about this movie.",
]

for texto in textos:
    resultado = clasificador(texto)
    etiqueta = resultado[0]["label"]
    confianza = resultado[0]["score"]
    print(f"   [{etiqueta:>4} {confianza:.3f}] {texto[:50]}...")

print("\n3. Clasificacion multiple (batch):")
resultados = clasificador(textos, batch_size=2)
for texto, resultado in zip(textos, resultados):
    print(f"   [{resultado['label']:>4}] {texto[:50]}...")

   [POSITIVE 1.000] This movie was absolutely fantastic! I loved every...
   [NEGATIVE 1.000] This is the worst film I have ever seen in my enti...
   [NEGATIVE 0.998] The acting was okay but the plot was confusing....
   [NEGATIVE 0.999] I am not sure how to feel about this movie....

3. Clasificacion multiple (batch):


   [POSITIVE] This movie was absolutely fantastic! I loved every...
   [NEGATIVE] This is the worst film I have ever seen in my enti...
   [NEGATIVE] The acting was okay but the plot was confusing....
   [NEGATIVE] I am not sure how to feel about this movie....


## Detrás del pipeline: tokenizer + modelo manual

In [3]:
tokenizer = AutoTokenizer.from_pretrained(
    "distilbert-base-uncased-finetuned-sst-2-english"
)
modelo = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased-finetuned-sst-2-english"
)

texto = "I love this product!"
print(f"   Texto: '{texto}'")

inputs = tokenizer(texto, return_tensors="pt")
print(f"   Input IDs: {inputs['input_ids'].tolist()}")
print(f"   Attention mask: {inputs['attention_mask'].tolist()}")

with torch.no_grad():
    outputs = modelo(**inputs)

logits = outputs.logits
print(f"\n   Logits (valores crudos del modelo): {logits.tolist()}")

probabilidades = F.softmax(logits, dim=-1)
print(f"   Probabilidades (softmax): {probabilidades.tolist()}")

labels = ["NEGATIVE", "POSITIVE"]
for i, label in enumerate(labels):
    print(f"   {label}: {probabilidades[0][i]:.4f} ({probabilidades[0][i]*100:.1f}%)")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 16230.98it/s]

   Texto: 'I love this product!'
   Input IDs: [[101, 1045, 2293, 2023, 4031, 999, 102]]
   Attention mask: [[1, 1, 1, 1, 1, 1, 1]]

   Logits (valores crudos del modelo): [[-4.359877109527588, 4.715570449829102]]
   Probabilidades (softmax): [[0.00011442836694186553, 0.9998855590820312]]
   NEGATIVE: 0.0001 (0.0%)
   POSITIVE: 0.9999 (100.0%)


## Comparación de confianza entre textos

In [4]:
textos_prueba = [
    "I love this, it's amazing!",
    "This is terrible, I hate it.",
    "The weather today is cloudy.",
]

for t in textos_prueba:
    r = clasificador(t)
    label = r[0]["label"]
    score = r[0]["score"]
    max_label = "ALTA" if score > 0.95 else "MEDIA" if score > 0.80 else "BAJA"
    print(f"   [{label:>4} conf={score:.3f} ({max_label})] {t}")

   [POSITIVE conf=1.000 (ALTA)] I love this, it's amazing!
   [NEGATIVE conf=1.000 (ALTA)] This is terrible, I hate it.
   [NEGATIVE conf=0.991 (ALTA)] The weather today is cloudy.


## Explorando el vocabulario

In [5]:
tok = AutoTokenizer.from_pretrained(
    "distilbert-base-uncased-finetuned-sst-2-english"
)
palabras = ["love", "hate", "good", "bad", "amazing", "terrible", "[UNK]"]
for palabra in palabras:
    token_id = tok.encode(palabra, add_special_tokens=False)
    print(f"   '{palabra}' -> token ID: {token_id}")

   'love' -> token ID: [2293]
   'hate' -> token ID: [5223]
   'good' -> token ID: [2204]
   'bad' -> token ID: [2919]
   'amazing' -> token ID: [6429]
   'terrible' -> token ID: [6659]
   '[UNK]' -> token ID: [100]


## Conclusión

- `pipeline` abstrae todo: tokenizar, modelar, postprocesar
- A mano vemos las piezas: **tokenizer** → **modelo** → **logits** → **softmax**
- La **confianza** varía: textos claros dan >0.95; los ambiguos bajan